# Autonomous Steering Prediction from Simulator Camera Frames

## Project overview

This personal project explores end-to-end steering-angle prediction from front-facing driving simulator images. I use the Udacity self-driving car simulator data to compare three modeling approaches:

1. an image-only CNN baseline,
2. a sequence-aware C-LSTM model that uses recent visual history, and
3. a simplified Behavior Transformer-style classification extension.

The main question is whether temporal context improves steering prediction on visually different driving environments, especially when the model is evaluated on the Jungle track.

## References used

- [End to End Learning for Self-Driving Cars](https://images.nvidia.com/content/tegra/automotive/images/2016/solutions/pdf/end-to-end-dl-using-px.pdf)
- [End-to-End Deep Learning for Steering Autonomous Vehicles Considering Temporal Dependencies](https://arxiv.org/abs/1710.03804)
- [Behavior Transformers: Cloning k modes with one stone](https://proceedings.neurips.cc/paper_files/paper/2022/file/90d17e882adbdda42349db6f50123817-Paper-Conference.pdf)


## 1. Data exploration and diagnostics

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import torch

In [ ]:
zip_path = "udacity_data.zip"
data_dir = "./udacity_data"

The notebook expecontinuous `udacity_data.zip` to be in the working directory. If the dataset has already been extracted, the next cell skips the extraction step.

In [ ]:
if not os.path.exists(data_dir):
    print(f"Extracting {zip_path}...")
    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        print("Extraction complete.")
    except FileNotFoundError:
        print(f"Could not find {zip_path}. Place the archive in the working directory and rerun this cell.")
        raise
else:
    print(f"Data already extracted to {data_dir}. Skipping extraction.")

## Dataset fields

The dataset contains camera image paths and vehicle-control signals for each simulator frame.

| Field | Description |
|---|---|
| `center` | Path to the center camera image |
| `left` | Path to the left camera image |
| `right` | Path to the right camera image |
| `steering` | Steering angle at the frame; `0` means straight, positive means right turn, negative means left turn |
| `throttle` | Acceleration command |
| `brake` | Brake command |
| `speed` | Vehicle speed |
| `track_source` | Track folder used to identify the source environment |

Because the data were collected in a keyboard-controlled simulator, throttle and brake are expected to behave differently from real pedal signals. In practice, throttle is almost always close to `1`, while brake is always `0`, so the camera frames and steering labels are the main useful signals for this project.


In [ ]:
def set_seed(seed: int = 1746):
    """Set the main random seeds used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
columns = ["center", "left", "right", "steering", "throttle", "brake", "speed"]
folders_to_load = ["self_driving_car_dataset_make", "self_driving_car_dataset_jungle"]

all_dataframes = []

for folder_name in folders_to_load:
    csv_path = os.path.join(data_dir, folder_name, "driving_log.csv")

    if os.path.exists(csv_path):
        print(f"Loading data from: {folder_name}")
        temp_df = pd.read_csv(csv_path, names=columns)

        def clean_path_for_folder(path, current_folder=folder_name):
            filename = path.split("/")[-1].split("\\")[-1]
            return os.path.join(data_dir, current_folder, "IMG", filename)

        for camera in ["center", "left", "right"]:
            temp_df[camera] = temp_df[camera].apply(clean_path_for_folder)

        temp_df["track_source"] = folder_name
        all_dataframes.append(temp_df)
    else:
        print(f"Missing driving_log.csv in {folder_name}")

df = pd.concat(all_dataframes, ignore_index=True)
print(f"Total frames loaded: {len(df)}")

### Dataset sanity checks

Before modeling, I check the numerical variables for scale, missing variation, and near-constant behavior. This helps decide which signals are useful as model inputs or diagnostics.

In [ ]:
continuous_vars = ["steering", "throttle", "brake", "speed"]

print(df.shape)

summary = df[continuous_vars].describe().T[[
    "count", "mean", "std", "min", "25%", "50%", "75%", "max"
]]

summary = summary.rename(columns={
    "count": "Count",
    "mean": "Mean",
    "std": "SD",
    "min": "Min",
    "25%": "25%",
    "50%": "50%",
    "75%": "75%",
    "max": "Max"
})

print("\nsummary stats:")
display(summary.round(4))

brake_zero_count = (df["brake"] == 0).sum()
throttle_one_count = (df["throttle"] == 1.0).sum()

print("\n'brake' value counts:")
print(df["brake"].value_counts().sort_index())
print(f"All brake values are zero: {brake_zero_count == len(df)}")
print(f"# of brake values equal to 0: {brake_zero_count}")

if brake_zero_count == len(df):
    print("`brake` is identical across all entries: every value is 0.")

print("\n'throttle' top 10 value counts:")
print(df["throttle"].value_counts().head(10))
print(f"Number of throttle values equal to 1.0: {throttle_one_count}")
print(f"% of throttle values equal to 1.0: {throttle_one_count / len(df):.4f}")

if throttle_one_count == 7320:
    print("`throttle` is almost identical across entries: 7320 of 7334 values are exactly 1.0.")

print("\nMeans of continuous variables:")
print(df[continuous_vars].mean().round(4))

The combined dataset contains **7,334 frames**. The continuous variables show the following pattern:

- `steering` has meaningful variation, although many values are exactly `0`.
- `brake` is constant at `0` for every row.
- `throttle` is nearly constant: 7,320 of 7,334 rows are exactly `1.0`.
- `speed` is usually close to 30, with limited variation.

For this reason, I treat the image frames as the main predictor and steering angle as the target. Throttle and brake are not useful learning targets in this version because they contain almost no information.

### Steering-angle imbalance

The steering labels are heavily concentrated around zero. This matters because a model trained directly on the raw distribution can reduce loss by overpredicting straight driving.

In [ ]:
tot_frames = len(df)

straight_frames = (df['steering'] == 0.0).sum()

straight_percentage = (straight_frames / tot_frames) * 100

print(f"straight: {straight_frames}")
print(f"total: {tot_frames}")
print(f"Percentage of straight: {straight_percentage:.2f}%")

A large share of frames are straight-driving examples. If I train directly on this distribution, the model can learn a conservative strategy: predict steering values close to `0` most of the time. That would reduce average error on the training set but perform poorly on curves, where under-steering or drifting becomes more likely.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(df['steering'], bins=50, edgecolor='black')
plt.title('Distribution of Steering Angles')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')
plt.show()

In [ ]:
def flatten_distribution(dataframe, drop_probability):
    """Downsample exact-zero steering frames while keeping turning frames."""
    filtered_rows = []

    for _, row in dataframe.iterrows():
        steering_angle = row["steering"]

        if steering_angle == 0.0 and random.random() <= drop_probability:
            continue

        filtered_rows.append(row)

    return pd.DataFrame(filtered_rows)


set_seed(1746)

balanced_df = flatten_distribution(df, drop_probability=0.95)
print("Balanced dataset:", balanced_df.shape)

balanced_total = len(balanced_df)
balanced_straight = (balanced_df["steering"] == 0.0).sum()

print(f"# of total frames: {balanced_total}")
print(f"# of straight frames: {balanced_straight}")

plt.figure(figsize=(10, 6))
plt.hist(
    balanced_df["steering"],
    bins=50,
    edgecolor="black"
)
plt.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Zero steering"
)
plt.title("Steering Angle Distribution After Downsampling")
plt.xlabel("Steering angle")
plt.ylabel("Number of frames")
plt.legend()
plt.show()

To reduce the dominance of straight-driving frames, I downsampled a random 95% of observations where the steering angle is exactly `0`. This does not fully balance the dataset, but it makes the turning examples more visible during analysis and training.

### Track-domain comparison

The dataset combines frames from two visually different tracks. I compare a random image from each track to understand the domain shift that the model needs to handle.

In [ ]:
make_data = df[df["track_source"] == "self_driving_car_dataset_make"]
jungle_data = df[df["track_source"] == "self_driving_car_dataset_jungle"]

set_seed(1746)

make_sample = make_data.sample(1).iloc[0]
jungle_sample = jungle_data.sample(1).iloc[0]

img_make = mpimg.imread(make_sample["center"])
img_jungle = mpimg.imread(jungle_sample["center"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Visual Difference Between Tracks", fontsize=16)

axes[0].imshow(img_make)
axes[0].set_title("Make / Lake track")
axes[0].axis("off")

axes[1].imshow(img_jungle)
axes[1].set_title("Jungle track")
axes[1].axis("off")

plt.tight_layout()
plt.show()

The Make/Lake track appears brighter and more structured, with clearer lane boundaries and flatter visual geometry. The Jungle track is visually more cluttered, with heavier vegetation, more shadows, and a less uniform road surface. This makes the Jungle track a useful stress test for generalization.

### Side-camera recovery targets

The simulator includes left, center, and right camera images. I use the side cameras as a simple recovery-data strategy: left-camera frames are paired with a slightly more positive steering target, and right-camera frames are paired with a slightly more negative steering target.

In [ ]:
straight_driving = df[abs(df["steering"]) < 0.05]

set_seed(1746)
sample = straight_driving.sample(1).iloc[0]

img_left = mpimg.imread(sample["left"])
img_center = mpimg.imread(sample["center"])
img_right = mpimg.imread(sample["right"])

original_steering = sample["steering"]
correction = 0.2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f"Side-Camera Recovery Targets (Original Steering: {original_steering:.2f})",
    fontsize=16
)

axes[0].imshow(img_left)
axes[0].set_title(f"Left camera\nTarget steering: {original_steering + correction:.2f}")
axes[0].axis("off")

axes[1].imshow(img_center)
axes[1].set_title(f"Center camera\nTarget steering: {original_steering:.2f}")
axes[1].axis("off")

axes[2].imshow(img_right)
axes[2].set_title(f"Right camera\nTarget steering: {original_steering - correction:.2f}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

For a center-frame steering label of \(\theta\), I assign:

- center camera: \(\theta\)
- left camera: \(\theta + 0.2\)
- right camera: \(\theta - 0.2\)

This gives the model examples that mimic recovery from being offset within the lane.

## 2. Image-only CNN baseline

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

### Train, validation, and test split

I reserve 500 frames from each track for final visualization. The remaining Make/Lake frames and part of the Jungle frames are used for training, while the rest of the Jungle frames are used for validation. This makes validation intentionally harder than training because it emphasizes the more visually complex track.

In [ ]:
RANDOM_STATE = 1746
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 0.001
STEERING_CORRECTION = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
make_df = df[df["track_source"] == "self_driving_car_dataset_make"].copy()
jungle_df = df[df["track_source"] == "self_driving_car_dataset_jungle"].copy()

# Randomly select 500 images from each track for testing
make_trainable, make_test_df = train_test_split(
    make_df,
    test_size=500,
    random_state=RANDOM_STATE,
    shuffle=True
)

jungle_trainable, jungle_test_df = train_test_split(
    jungle_df,
    test_size=500,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Randomly split the remaining jungle images into training and validation
jungle_train, jungle_val = train_test_split(
    jungle_trainable,
    test_size=0.5,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_df = pd.concat([make_trainable, jungle_train], ignore_index=True)
val_df = jungle_val.copy()

print(f"Total original frames: {len(df)}")
print(f"Training frames (Make + random Jungle subset): {len(train_df)}")
print(f"Validation frames (random Jungle subset): {len(val_df)}")
print(f"Test frames held out for final graphs: {len(make_test_df) + len(jungle_test_df)}")

### Custom PyTorch dataset

The dataset class loads center, left, and right camera images, applies steering correction for recovery samples, and uses horizontal flipping as a symmetry-based augmentation.

In [ ]:
class DrivingDataset(Dataset):
    def __init__(self, dataframe, is_training=True, correction=0.2):
        self.dataframe = dataframe
        self.is_training = is_training
        self.correction = correction
        self.all_samples = list(zip(
            dataframe["center"],
            dataframe["left"],
            dataframe["right"],
            dataframe["steering"]
        ))
        self.active_samples = self.all_samples

    def prepare_epoch(self):
        """Refresh the training sample list and reduce zero-steering dominance."""
        if self.is_training:
            self.active_samples = []

            for sample in self.all_samples:
                if sample[3] == 0.0 and random.random() > 0.5:
                    continue

                self.active_samples.append(sample)

            random.shuffle(self.active_samples)

    def __len__(self):
        return len(self.active_samples)

    def __getitem__(self, index):
        center_path, left_path, right_path, steering = self.active_samples[index]

        img_center = cv2.cvtColor(cv2.imread(center_path), cv2.COLOR_BGR2RGB)
        img_left = cv2.cvtColor(cv2.imread(left_path), cv2.COLOR_BGR2RGB)
        img_right = cv2.cvtColor(cv2.imread(right_path), cv2.COLOR_BGR2RGB)

        images = [img_center, img_left, img_right]
        angles = [
            steering,
            steering + self.correction,
            steering - self.correction
        ]

        final_images, final_angles = [], []

        for img, angle in zip(images, angles):
            final_images.append(img)
            final_angles.append(angle)

            if self.is_training and random.random() > 0.5:
                final_images.append(cv2.flip(img, 1))
                final_angles.append(angle * -1.0)

        tensor_images = [
            torch.from_numpy(img.transpose(2, 0, 1)).float()
            for img in final_images
        ]
        tensor_angles = torch.tensor(final_angles, dtype=torch.float32)

        return torch.stack(tensor_images), tensor_angles


def custom_collate(batch):
    """Flatten augmented samples into one image batch."""
    images = torch.cat([item[0] for item in batch], dim=0)
    angles = torch.cat([item[1] for item in batch], dim=0)

    return images, angles


train_dataset = DrivingDataset(
    train_df,
    is_training=True,
    correction=STEERING_CORRECTION
)

val_dataset = DrivingDataset(
    val_df,
    is_training=False,
    correction=STEERING_CORRECTION
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=custom_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=custom_collate
)

The custom dataset expands each driving-log row into multiple training examples. For each frame, it uses the center camera image plus the left and right camera images with corrected steering targets. During training, it also randomly flips images and multiplies the steering label by `-1`, which adds left-right symmetry without collecting new data.

The `custom_collate` function flattens those augmented examples into a standard image batch with shape `(batch, channels, height, width)`. This keeps the training loop simple while still using multiple camera views per log row.

### CNN model design

The CNN uses a compact convolutional feature extractor followed by a small regression head. The input image is normalized, cropped to remove less-informative sky and hood regions, and then mapped to a single continuous steering-angle prediction.

In [ ]:
# CNN feature extractor and regression head.
class DrivingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, kernel_size=5, stride=2), nn.ReLU(), nn.Dropout2d(p=0.10),
            nn.Conv2d(36, 48, kernel_size=5, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, kernel_size=3, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), nn.ReLU()
        )
        
        # flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 160, 320)
            dummy = dummy[:, :, 70:-24, :] # crop 
            dummy = dummy / 255.0 - 0.5   
            features = self.feature_extractor(dummy)
            flattened_size = features.view(1, -1).shape[1]
        
        self.regression_head = nn.Sequential(
            nn.Linear(flattened_size, 100),
            nn.ReLU(),
            nn.Dropout(p=0.30),
            
            nn.Linear(100, 50),
            nn.ReLU(),
            
            nn.Linear(50, 1) # continuous steering output
        )
        
    def forward(self, x):
        x = x / 255.0 - 0.5 # normalize inputs
        x = x[:, :, 70:-24, :] # crop 
        x = self.feature_extractor(x) # cnn 
        x = x.view(x.size(0), -1) # flatten
        x = self.regression_head(x) # regression
        
        return x.squeeze(1) # return one prediction per image

# Initialize model.
model = DrivingCNN().to(device)

# Count trainable parameters.
para = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"# of trainable parameters: {para}")

### CNN training and validation

In [ ]:
# Train the CNN baseline and validate on Jungle frames.

# Training and validation datasets.
train_dataset = DrivingDataset(
    train_df,
    is_training=True,
    correction=STEERING_CORRECTION
)

val_dataset = DrivingDataset(
    val_df,
    is_training=False,
    correction=STEERING_CORRECTION
)

# Data loaders.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=custom_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=custom_collate
)

# Steering prediction is treated as a regression problem.
criterion = nn.MSELoss()

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Store loss values for comparison.
train_losses = []
val_losses = []

# Training loop
for epoch in range(EPOCHS):
    # Prepare model for training.
    model.train()
    train_dataset.prepare_epoch() 
    running_train_loss = 0.0

    # Training batches.
    for images, angles in train_loader:
        # Move tensors to the selected device.
        images = images.to(device)
        angles = angles.to(device)
        # Clear old gradients.
        optimizer.zero_grad()
        # Predict steering angles.
        predictions = model(images)
        # Compute MSE loss.
        loss = criterion(predictions, angles)
        # Backpropagation.
        loss.backward()
        # Update model weights.
        optimizer.step()

        running_train_loss += loss.item()

    # Average training loss for this epoch.
    avg_train_loss = running_train_loss / len(train_loader)

    train_losses.append(avg_train_loss)

    # Validation mode.
    model.eval()

    running_val_loss = 0.0

    # Disable gradients during validation.
    with torch.no_grad():

        # Validation batches.
        for images, angles in val_loader:

            images = images.to(device)
            angles = angles.to(device)

            # Predict steering angles.
            predictions = model(images)

            # Compute validation loss.
            loss = criterion(predictions, angles)

            running_val_loss += loss.item()

    # average validation loss
    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Print epoch summary.
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Training Loss: {avg_train_loss:.6f}")
    print(f"Validation Loss (Jungle Only): {avg_val_loss:.6f}")

# Print full loss history.
print("train_losses:", train_losses)
print("val_losses:", val_losses)

The CNN was evaluated on held-out Make/Lake frames and held-out Jungle frames. It performed better on the Make/Lake environment, which is visually closer to much of the training distribution. Performance dropped on Jungle frames, suggesting that visual domain shift is a major source of error.

## 3. Sequence model: C-LSTM

The next model adds temporal memory. Instead of predicting steering from a single image, the C-LSTM processes a short sequence of consecutive frames. The CNN acontinuous as a spatial encoder for each frame, and the LSTM models how those visual features evolve over time.

In [ ]:
DATA_DIR = "./udacity_data"
FOLDERS = ['self_driving_car_dataset_make', 'self_driving_car_dataset_jungle']
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 0.001
SEQ_LENGTH = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
print("Loading CSV data...")
columns = ['center', 'left', 'right', 'steering', 'throttle', 'brake', 'speed']
all_dataframes = []

for folder in FOLDERS:
    csv_path = os.path.join(DATA_DIR, folder, 'driving_log.csv')
    if os.path.exists(csv_path):
        temp_df = pd.read_csv(csv_path, names=columns)
        
        def clean_path(path, current_folder=folder):
            filename = path.split('/')[-1].split('\\')[-1]
            return os.path.join(DATA_DIR, current_folder, 'IMG', filename)

        temp_df['center'] = temp_df['center'].apply(clean_path)
        temp_df['track_source'] = folder
        all_dataframes.append(temp_df)

df = pd.concat(all_dataframes, ignore_index=True)
val_df = df[df['track_source'] == 'self_driving_car_dataset_jungle'].copy()
train_df, _ = train_test_split(df, test_size=0.2, \
                               random_state=42, \
                               shuffle=False)  # Preserve temporal order for this split.

### Sequence dataset

In [ ]:
# Sequence dataset for C-LSTM.

class DrivingSequenceDataset(Dataset):
    def __init__(self, dataframe, seq_length=5, is_training=True):
        self.dataframe = dataframe.copy()
        self.seq_length = seq_length
        self.is_training = is_training
        
        # Preserve time order within each track.
        if 'original_index' not in self.dataframe.columns:
            self.dataframe['original_index'] = self.dataframe.index
        self.dataframe = self.dataframe.sort_values(
            by=['track_source', 'original_index']
        ).reset_index(drop=True)
        self.all_sequences = []
        
        # Build continuous sequences by track.
        for track in self.dataframe['track_source'].unique():
            track_df = self.dataframe[self.dataframe['track_source'] 
                == track].reset_index(drop=True)
            for i in range(self.seq_length - 1, len(track_df)):
                sequence_rows = track_df.iloc[i - self.seq_length + 1 : i + 1]
                self.all_sequences.append(sequence_rows)
        self.active_sequences = self.all_sequences

    def prepare_epoch(self):
        # Reduce zero-steering bias.
        if self.is_training:
            self.active_sequences = []
            for seq in self.all_sequences:
                final_steering = seq.iloc[-1]['steering']
                if final_steering == 0.0 and random.random() > 0.5:
                    continue
                self.active_sequences.append(seq)
            random.shuffle(self.active_sequences)

    def __len__(self):
        return len(self.active_sequences)

    def __getitem__(self, index):
        sequence_rows = self.active_sequences[index]
        images = []
        
        for _, row in sequence_rows.iterrows():
            img = cv2.imread(row['center'])
            
            if img is None:
                raise FileNotFoundError(f"Image not found: {row['center']}")
            
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
        
        final_steering = sequence_rows.iloc[-1]['steering']
        
        # Horizontal flip augmentation.
        if self.is_training and random.random() > 0.5:
            images = [cv2.flip(img, 1) for img in images]
            final_steering = final_steering * -1.0 # flip
        
        # Convert images to tensors.
        tensor_images = [
            torch.from_numpy(img.transpose(2, 0, 1)).float()
            for img in images
        ]
        # Shape: (sequence, channels, height, width).
        sequence_tensor = torch.stack(tensor_images)
        label = torch.tensor(final_steering, dtype=torch.float32)
        return sequence_tensor, label

# Sequence datasets.
seq_train_dataset = DrivingSequenceDataset(
    train_df,
    seq_length=SEQ_LENGTH,
    is_training=True
)

seq_val_dataset = DrivingSequenceDataset(
    val_df,
    seq_length=SEQ_LENGTH,
    is_training=False
)

# Data loaders.
seq_train_loader = DataLoader(
    seq_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)
seq_val_loader = DataLoader(
    seq_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Inspect one mini-batch.
seq_train_dataset.prepare_epoch()

sample_sequences, sample_labels = next(iter(seq_train_loader))

print("sequence batch shape:", sample_sequences.shape)
print("label batch shape:", sample_labels.shape)

### C-LSTM architecture

This model reuses the CNN feature extractor as a frame encoder, then sends the frame-level feature vectors into an LSTM. The regression head predicontinuous the steering angle for the final frame in the sequence.

In [ ]:
# C-LSTM architecture.

class CLSTMDrivingModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.spatial_encoder = model.feature_extractor
        
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 160, 320)
            dummy = dummy / 255.0 - 0.5 # normalize
            dummy = dummy[:, :, 70:-24, :] # crop
            features = self.spatial_encoder(dummy)
            self.feature_size = features.view(1, -1).shape[1] # feature size
        
        self.lstm = nn.LSTM(
            input_size=self.feature_size,
            hidden_size=128,
            batch_first=True
        )
        
        self.regression_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.30),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        # (batch, seq, channels, height, width)
        batch_size, seq_length, C, H, W = x.shape
        
        feature_sequence = []
        
        # Process each frame.
        for t in range(seq_length):
            frame = x[:, t, :, :, :]
            frame = frame / 255.0 - 0.5
            frame = frame[:, :, 70:-24, :]
            features = self.spatial_encoder(frame)
            features = features.view(batch_size, -1)
            
            feature_sequence.append(features)
        
        feature_sequence = torch.stack(feature_sequence, dim=1) # (batch, seq, feature_size)
        
        lstm_output, _ = self.lstm(feature_sequence) # temporal decoding
        
        final_output = lstm_output[:, -1, :]
        
        steering = self.regression_head(final_output) # prediction
        
        return steering.squeeze(1)


# Initialize C-LSTM model.
clstm_model = CLSTMDrivingModel().to(device)

# Count trainable parameters.
clstm_para = sum(
    p.numel() for p in clstm_model.parameters() if p.requires_grad
)

print(clstm_model)
print(f"Number of trainable parameters: {clstm_para}")

### Model comparison

I train the C-LSTM using the same mean-squared-error objective as the CNN and compare their validation losses on the Jungle track.

In [ ]:
# Quantitative Comparison 

# Train the C-LSTM model.
clstm_criterion = nn.MSELoss()

clstm_optimizer = optim.Adam(
    clstm_model.parameters(),
    lr=LEARNING_RATE
)

# Store losses for comparison.
clstm_train_losses = []
clstm_val_losses = []

for epoch in range(EPOCHS):

    # Training mode.
    clstm_model.train()

    # Refresh filtered training sequences.
    seq_train_dataset.prepare_epoch()

    running_train_loss = 0.0

    # loop through training batches
    for sequences, labels in seq_train_loader:

        # Move tensors to the selected device.
        sequences = sequences.to(device)
        labels = labels.to(device)

        # clear previous gradients
        clstm_optimizer.zero_grad()

        # Predict steering angles.
        predictions = clstm_model(sequences)

        # Compute MSE loss.
        loss = clstm_criterion(predictions, labels)

        # Backpropagation.
        loss.backward()

        # update weights
        clstm_optimizer.step()

        running_train_loss += loss.item()

    # Average training loss.
    avg_train_loss = running_train_loss / len(seq_train_loader)

    clstm_train_losses.append(avg_train_loss)

    # Validation mode.
    clstm_model.eval()

    running_val_loss = 0.0

    # Disable gradients during validation.
    with torch.no_grad():

        # Validation batches.
        for sequences, labels in seq_val_loader:

            sequences = sequences.to(device)
            labels = labels.to(device)

            # Predict steering angles.
            predictions = clstm_model(sequences)

            # Compute validation loss.
            loss = clstm_criterion(predictions, labels)

            running_val_loss += loss.item()

    # average validation loss
    avg_val_loss = running_val_loss / len(seq_val_loader)

    clstm_val_losses.append(avg_val_loss)

    # Print epoch summary.
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"C-LSTM Train Loss: {avg_train_loss:.6f}")
    print(f"C-LSTM Jungle Validation Loss: {avg_val_loss:.6f}")
    print("-" * 50)

# Compare final validation MSE.
cnn_final_val_mse = val_losses[-1]
clstm_final_val_mse = clstm_val_losses[-1]

print(f"Final CNN validation MSE on Jungle: {cnn_final_val_mse:.6f}")
print(f"Final C-LSTM validation MSE on Jungle: {clstm_final_val_mse:.6f}")

# Compare final validation errors.
if clstm_final_val_mse < cnn_final_val_mse:
    print("C-LSTM improved validation performance compared with the CNN.")
else:
    print("C-LSTM did not improve validation MSE compared with the CNN.")

### Why temporal context helps

The final validation MSE of the CNN on the Jungle track was `0.107769`, while the final validation MSE of the C-LSTM was `0.085311`. In this run, the C-LSTM reduced prediction error compared with the image-only CNN.

The improvement is plausible because driving is not a collection of independent images. Consecutive frames provide information about the vehicle’s recent path, upcoming curvature, and visual motion. The CNN sees each frame separately, while the C-LSTM uses recent frame history to produce smoother and more context-aware steering predictions.

In [ ]:
# Trace analysis on 300 continuous Jungle frames.

# Use a continuous block of Jungle validation data.
jungle_block = val_df.sort_index().reset_index(drop=True).iloc[:300].copy()

# CNN predictions.
cnn_truth = []
cnn_preds = []

model.eval()

with torch.no_grad():

    for i in range(len(jungle_block)):

        row = jungle_block.iloc[i]

        img = cv2.imread(row['center'])

        if img is None:
            raise FileNotFoundError(row['center'])

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_tensor = torch.from_numpy(
            img.transpose(2, 0, 1)
        ).float()

        img_tensor = img_tensor.unsqueeze(0).to(device)

        pred = model(img_tensor).item()

        cnn_preds.append(pred)
        cnn_truth.append(row['steering'])


# C-LSTM predictions.
clstm_preds = [np.nan] * (SEQ_LENGTH - 1)

clstm_model.eval()

with torch.no_grad():

    for i in range(SEQ_LENGTH - 1, len(jungle_block)):

        sequence_rows = jungle_block.iloc[
            i - SEQ_LENGTH + 1 : i + 1
        ]

        images = []

        for _, row in sequence_rows.iterrows():

            img = cv2.imread(row['center'])

            if img is None:
                raise FileNotFoundError(row['center'])

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            img_tensor = torch.from_numpy(
                img.transpose(2, 0, 1)
            ).float()

            images.append(img_tensor)

        sequence_tensor = torch.stack(images)

        sequence_tensor = sequence_tensor.unsqueeze(0).to(device)

        pred = clstm_model(sequence_tensor).item()

        clstm_preds.append(pred)


# Plot predictions 
plt.figure(figsize=(14, 6))

plt.plot(cnn_truth, label="Ground Truth Steering")
plt.plot(cnn_preds, label="CNN prediction")
plt.plot(clstm_preds, label="C-LSTM prediction")

plt.title("Trace Analysis: CNN vs. C-LSTM on 300 Jungle Frames")
plt.xlabel("Frame index")
plt.ylabel("Steering angle")

plt.legend()
plt.show()

In the 300-frame trace plot, the ground truth drops sharply around frame 50. The CNN prediction reacontinuous more gradually and does not fully match the sudden change. The C-LSTM prediction is still imperfect, but it follows the broad downward trend more smoothly.

This behavior matches the model design: the CNN extracontinuous visual features from each frame independently, while the LSTM integrates those features over time. The sequence model can therefore use recent road context instead of relying only on the current frame.

The sequence order is essential. If the frames were shuffled, the LSTM would receive an incorrect driving history. Its memory gates would update using unrelated frames, so the model would no longer represent a coherent road trajectory. That would remove much of the smoothing and temporal-context benefit that makes the C-LSTM useful.

### Architecture interpretation

The C-LSTM model combines a CNN with an LSTM so that the prediction uses both spatial and temporal information.

The CNN portion works as a **spatial encoder**. Each image frame is cropped, normalized, and passed through convolutional layers. The output is a compact feature vector that summarizes visual cues such as road boundaries, lane curvature, and surrounding scenery.

The LSTM portion works as a **temporal decoder**. It receives a sequence of feature vectors in chronological order and maintains a hidden state across frames. Through its input, forget, and output gates, the LSTM decides which past information should be retained or discarded. The final LSTM state is then passed through a regression head that predicontinuous the steering angle for the last frame in the sequence.

This design is better suited to driving than a single-frame CNN because steering decisions usually depend on recent motion and road direction, not just one still image.

## 4. Optional extension: simplified Behavior Transformer

As an additional experiment, I test a simplified Behavior Transformer-style idea. Instead of predicting steering as a continuous value with mean squared error, I discretize the steering labels into action bins and train a classifier.

This version is intentionally simplified: it keeps the CNN backbone and replaces the regression output with a 10-class steering-bin classifier. The goal is not to reproduce the full BeT paper, but to understand what happens when continuous control is represented as a small action vocabulary.

### Discrete action experiment

I cluster the training steering angles into `k = 10` action centers using k-means. Each training label is mapped to the nearest steering center, and the CNN is trained with cross-entropy loss to classify each frame into one of those 10 bins.

For inference, the predicted class is mapped back to its corresponding steering-center value. I then compare the resulting discrete steering trace with the continuous ground-truth trace over a 300-frame Jungle segment.

In [ ]:
# Simplified Behavior Transformer experiment
from sklearn.cluster import KMeans

K = 10

# Extract steering labels.
steering_values = train_df['steering'] \
    .values.reshape(-1, 1)

# Cluster steering labels into discrete action centers.
kmeans = KMeans(
    random_state=1746,
    n_clusters=K,
    n_init=10
)

kmeans.fit(steering_values)

# Sort centers for interpretability.
action_centers = np.sort(
    kmeans.cluster_centers_.flatten()
)

print("10 Steering Action Centers:")
print(action_centers)


# Map a steering angle to the nearest action bin.
def steering_to_bin(angle, centers):

    diff = np.abs(centers - angle)

    return np.argmin(diff)


# Dataset for discrete-action model.
class BeTDataset(Dataset):

    def __init__(
        self,
        dataframe,
        centers,
        is_training=True
    ):

        self.dataframe = dataframe.reset_index(drop=True)

        self.centers = centers

        self.is_training = is_training

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        steering = row['steering']

        img = cv2.imread(row['center'])

        if img is None:
            raise FileNotFoundError(row['center'])

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        # Random horizontal flip augmentation.
        if self.is_training and random.random() > 0.5:

            img = cv2.flip(img, 1)

            steering = steering * -1

        # Convert image to tensor.
        img_tensor = torch.from_numpy(
            img.transpose(2, 0, 1)
        ).float()

        # Convert steering value to a class label.
        label = steering_to_bin(
            steering,
            self.centers
        )

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return img_tensor, label


# Datasets.
bet_train_dataset = BeTDataset(
    train_df,
    action_centers,
    is_training=True
)

bet_val_dataset = BeTDataset(
    val_df,
    action_centers,
    is_training=False
)


# Data loaders.
bet_train_loader = DataLoader(
    bet_train_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=0
)

bet_val_loader = DataLoader(
    bet_val_dataset,
    shuffle=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)


# CNN classifier for discrete steering bins.
class BeTCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.feature_extractor = nn.Sequential(

            nn.Conv2d(
                3,
                24,
                kernel_size=5,
                stride=2
            ),

            nn.ReLU(),

            nn.Conv2d(
                24,
                36,
                kernel_size=5,
                stride=2
            ),

            nn.ReLU(),

            nn.Dropout2d(p=0.10),

            nn.Conv2d(
                36,
                48,
                kernel_size=5,
                stride=2
            ),

            nn.ReLU(),

            nn.Conv2d(
                48,
                64,
                kernel_size=3,
                stride=2
            ),

            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU()
        )

        # Infer the flattened feature size.
        with torch.no_grad():

            dummy = torch.zeros(
                1,
                3,
                160,
                320
            )

            dummy = dummy / 255.0 - 0.5

            dummy = dummy[:, :, 70:-24, :]

            features = self.feature_extractor(dummy)

            flattened_size = features.view(
                1,
                -1
            ).shape[1]

        self.classifier = nn.Sequential(

            nn.Linear(
                flattened_size,
                100
            ),

            nn.ReLU(),

            nn.Dropout(p=0.30),

            nn.Linear(
                100,
                50
            ),

            nn.ReLU(),

            # 10 output classes
            nn.Linear(50, K)
        )

    def forward(self, x):

        # normalize
        x = x / 255.0 - 0.5

        # crop sky and hood
        x = x[:, :, 70:-24, :]

        x = self.feature_extractor(x)

        x = x.view(x.size(0), -1)

        return self.classifier(x)


bet_model = BeTCNN().to(device)


# Loss and optimizer.
bet_criterion = nn.CrossEntropyLoss()

bet_optimizer = optim.Adam(
    bet_model.parameters(),
    lr=LEARNING_RATE
)

BET_EPOCHS = 3

bet_train_losses = []

bet_val_losses = []


# Training loop
for epoch in range(BET_EPOCHS):

    bet_model.train()

    running_train_loss = 0.0

    for images, labels in bet_train_loader:

        images = images.to(device)

        labels = labels.to(device)

        bet_optimizer.zero_grad()

        outputs = bet_model(images)

        loss = bet_criterion(
            outputs,
            labels
        )

        loss.backward()

        bet_optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = (
        running_train_loss /
        len(bet_train_loader)
    )

    bet_train_losses.append(avg_train_loss)

    # validation
    bet_model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for images, labels in bet_val_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = bet_model(images)

            loss = bet_criterion(
                outputs,
                labels
            )

            running_val_loss += loss.item()

    avg_val_loss = (
        running_val_loss /
        len(bet_val_loader)
    )

    bet_val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{BET_EPOCHS}")

    print(
        f"BeT Train Loss: {avg_train_loss:.6f}"
    )

    print(
        f"BeT Validation Loss: {avg_val_loss:.6f}"
    )

# trace analysis
jungle_block = val_df \
    .sort_index() \
    .reset_index(drop=True) \
    .iloc[:300] \
    .copy()

ground_truth = []

bet_predictions = []

bet_model.eval()

with torch.no_grad():

    for i in range(len(jungle_block)):

        row = jungle_block.iloc[i]

        img = cv2.imread(row['center'])

        if img is None:
            raise FileNotFoundError(row['center'])

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        img_tensor = torch.from_numpy(
            img.transpose(2, 0, 1)
        ).float()

        img_tensor = img_tensor \
            .unsqueeze(0) \
            .to(device)

        outputs = bet_model(img_tensor)

        predicted_bin = torch.argmax(
            outputs,
            dim=1
        ).item()

        predicted_steering = action_centers[
            predicted_bin
        ]

        bet_predictions.append(
            predicted_steering
        )

        ground_truth.append(
            row['steering']
        )

In [ ]:
# Simplified Behavior Transformer experiment
# Plot predictions
plt.figure(figsize=(14, 6))
plt.plot(
    ground_truth,
    label="Ground Truth Steering"
)
plt.plot(
    bet_predictions,
    label="BeT Discrete Prediction"
)
plt.title(
    "Simplified BeT Prediction vs. Ground Truth on Jungle Frames"
)
plt.xlabel("Frame index")
plt.ylabel("Steering angle")
plt.legend()
plt.show()

### Behavior Transformer result interpretation

The discrete prediction line looks visibly “steppy” because the model can only output one of 10 fixed steering centers. Mathematically, the continuous steering space is quantized into a finite set of bins, so many slightly different steering angles are forced to share the same output value. The prediction therefore stays flat while the class remains the same, then jumps when the predicted class changes.

That behavior is not ideal for driving because steering should change continuously. The trace shows why the residual-action head proposed in the Behavior Transformer paper is important. The discrete action bin can capture the approximate steering mode, but a residual offset is needed to make fine-grained continuous corrections within each bin. Without that residual component, the controller loses precision and produces unrealistic stepwise steering.